# Presentation LLMs — Notebook demo (khung)

- Nhóm: Muội, Phúc, Toàn (thực hiện), Thư (hỗ trợ). Môn Học sâu, UTH.
- Chạy trên **Kaggle, GPU T4, bật Internet**. Đặt `SMOKE = True` để chạy thử; `SMOKE = False` + **Save & Run All** để chạy chính thức.
- Quy ước (docs/rules/RULES_v2.md §2): mỗi cell code có markdown trước (mục đích, đầu vào, đầu ra), comment tiếng Việt, cell **Phân tích** sau output; tham số sinh luôn truyền tường minh.
- Code: Toàn (toàn bộ phần 0–10). Người phụ trách lý thuyết từng mục ghi trong tiêu đề phần, phải giải thích được code phần đó sau khi bàn giao.

> Khung: chỉ có tiêu đề các phần, chưa có code.

## Phần 0 — Setup (lý thuyết + code: Toàn)

- Nội dung: Cấu hình chung (SMOKE, seed, tên model), thư viện, môi trường; cách lấy Qwen2.5-1.5B-Instruct, Qwen2.5-1.5B, tokenizer GPT-2, UIT-VSFC (lấy ở đâu, lấy thế nào); ghi outputs/results/env.json
- Kết quả ghi vào: xem `project_description.md` §2.3

Gồm 8 cell code (0.1–0.8), theo `docs/plans/detail_plan_notebook_v1.md` phần 0. Log ghi vào `outputs/logs/<RUN_ID>/` (smoke: `outputs/_smoke/logs/`).

### 0.1 Cấu hình chung
- **Mục đích:** đặt mọi tham số ở một chỗ (luật `RULES_v2` §2.1); các phần sau chỉ đọc lại, không gõ lại.
- **Đầu vào:** không có.
- **Đầu ra:** các hằng số `SMOKE`, `SEED`, tên model, `N_TEST`; thư mục `DATA_DIR` (trên Kaggle: Dataset chứa `data/`), `OUT_DIR`, `RESULTS_DIR`, `FIGURES_DIR`, `LOG_DIR`; `RUN_ID` theo giờ Việt Nam.

In [14]:
# 0.1 Cấu hình chung: mọi tham số đặt ở đây, các phần khác chỉ đọc lại
import random
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import torch

SMOKE: bool = True  # True: chạy thử vài mẫu; False: chạy chính thức (Save & Run All)
SEED: int = 42

MODEL_INSTRUCT = "Qwen/Qwen2.5-1.5B-Instruct"  # model chính
MODEL_BASE = "Qwen/Qwen2.5-1.5B"               # bản base, chỉ tải ở phần 5
TOKENIZER_GPT2 = "openai-community/gpt2"       # chỉ lấy tokenizer, dùng ở phần 3
DTYPE = torch.float16                          # T4 không hỗ trợ bf16 bằng phần cứng

N_TEST = 10 if SMOKE else 500  # số câu UIT-VSFC dùng ở phần 7, 10

# Nhánh Parquet do Hugging Face tự chuyển đổi (script tải cũ không chạy với datasets >= 4.0)
UIT_VSFC_BASE = ("https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/"
                 "resolve/refs%2Fconvert%2Fparquet/default")

# Gốc repo: chạy từ thư mục notebook/ (máy nhóm) thì lùi một cấp; trên Kaggle là /kaggle/working
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
# Dữ liệu nhóm (data/*.json): trên máy là ROOT/data; trên Kaggle là Dataset gắn vào notebook (/kaggle/input/...).
# Dò theo tên file để không phụ thuộc tên Dataset.
KAGGLE_INPUT = Path("/kaggle/input")
_found = sorted(KAGGLE_INPUT.glob("**/tasks_prompts.json")) if KAGGLE_INPUT.exists() else []
DATA_DIR = _found[0].parent if _found else ROOT / "data"
OUT_DIR = ROOT / "outputs" / "_smoke" if SMOKE else ROOT / "outputs"  # smoke không ghi đè kết quả thật
RESULTS_DIR = OUT_DIR / "results"
FIGURES_DIR = OUT_DIR / "figures"

# Mã lần chạy = thời điểm bắt đầu (giờ Việt Nam) + chế độ, ví dụ 20261118_093000_normal
RUN_START = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh"))
RUN_ID = RUN_START.strftime("%Y%m%d_%H%M%S") + ("_smoke" if SMOKE else "_normal")
LOG_DIR = OUT_DIR / "logs" / RUN_ID

for d in (RESULTS_DIR, FIGURES_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Cố định seed cho random, numpy, torch (torch.manual_seed đặt cả seed cho GPU)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"RUN_ID  : {RUN_ID}")
print(f"OUT_DIR : {OUT_DIR.relative_to(ROOT)}")  # in đường dẫn tương đối, không lộ đường dẫn máy cá nhân
print(f"N_TEST  : {N_TEST}")
print(f"DATA_DIR: {'Kaggle Dataset ' + DATA_DIR.name if _found else DATA_DIR.relative_to(ROOT)}")  # không in tên tài khoản Kaggle

RUN_ID  : 20261116_030403_smoke
OUT_DIR : outputs/_smoke
N_TEST  : 10


### 0.2 Hàm ghi log
- **Mục đích:** mọi lần chạy có log theo `docs/templates/RUN_LOG_TEMPLATE_v2.md`: `run.log` cho cả lần chạy, `partNN_<tên>.log` cho từng phần.
- **Đầu vào:** `LOG_DIR`, `RUN_ID` (cell 0.1).
- **Đầu ra:** hàm `now_iso()`, `log(text, part)`; dòng `=== RUN START ... ===` trong `run.log`.

In [15]:
# 0.2 Hàm ghi log: ghi thêm từng dòng, không ghi đè
def now_iso() -> str:
    """Thời điểm hiện tại theo giờ Việt Nam, dạng ISO 8601 (đến giây)."""
    return datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).isoformat(timespec="seconds")


def log(text: str, part: str | None = None, to_run: bool = False, echo: bool = True) -> None:
    """Ghi một dòng log.

    - Không có `part`: ghi vào run.log (log chung của cả lần chạy).
    - Có `part` (vd "part00_setup"): ghi vào <part>.log; thêm `to_run=True` để ghi cả vào run.log
      (dùng cho thông tin đầu log và dòng tổng kết; dòng "mỗi mẫu" chỉ ghi vào log của phần).
    """
    files = []
    if part is None or to_run:
        files.append(LOG_DIR / "run.log")
    if part is not None:
        files.append(LOG_DIR / f"{part}.log")
    for f in files:
        with f.open("a", encoding="utf-8") as fh:  # "a" = ghi thêm vào cuối file
            fh.write(text + "\n")
    if echo:
        print(text)


log(f"=== RUN START {RUN_START.isoformat(timespec='seconds')} ===")
log(f"run_id: {RUN_ID} | mode: {'smoke' if SMOKE else 'normal'} (SMOKE={SMOKE}) | seed: {SEED} | dtype: {str(DTYPE).replace('torch.', '')}")
log(f"=== PART 00 setup START {now_iso()} ===", part="part00_setup")

# Tự kiểm: file log đã có và có dòng mở đầu
assert (LOG_DIR / "run.log").exists()
assert "RUN START" in (LOG_DIR / "run.log").read_text(encoding="utf-8")
assert (LOG_DIR / "part00_setup.log").exists()

=== RUN START 2026-11-16T03:04:03+07:00 ===
run_id: 20261116_030403_smoke | mode: smoke (SMOKE=True) | seed: 42 | dtype: float16
=== PART 00 setup START 2026-11-16T03:04:03+07:00 ===


### 0.3 Môi trường và thư viện
- **Mục đích:** in phiên bản thật của Python, thư viện, GPU (cô yêu cầu slide nêu thư viện và môi trường); không ghi theo trí nhớ.
- **Đầu vào:** môi trường Kaggle (Settings → Accelerator: GPU T4, Internet: On).
- **Đầu ra:** dict `ENV`; ghi vào log; cell 0.8 lưu thành `env.json`.

In [16]:
# 0.3 Môi trường: đọc phiên bản bằng importlib.metadata (không cần import cả thư viện)
import platform
from importlib.metadata import PackageNotFoundError, version


def pkg_version(name: str) -> str:
    """Phiên bản của một thư viện đã cài; trả "chưa cài" nếu không có."""
    try:
        return version(name)
    except PackageNotFoundError:
        return "chưa cài"


assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4"

ENV = {
    "run_id": RUN_ID,
    "date": now_iso(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),  # Kaggle T4 x2 → 2; model chỉ dùng GPU 0
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "dtype": str(DTYPE).replace("torch.", ""),
}
for lib in ["transformers", "accelerate", "pandas", "pyarrow", "scikit-learn", "statsmodels", "matplotlib"]:
    ENV[lib] = pkg_version(lib)

for k, v in ENV.items():
    log(f"{k}: {v}", part="part00_setup", to_run=True)

run_id: 20261116_030403_smoke
date: 2026-11-16T03:04:03+07:00
python: 3.13.16
torch: 2.11.0+cu130
cuda: 13.0
gpu: Tesla T4
gpu_count: 2
gpu_vram_gb: 14.6
dtype: float16
transformers: 5.20.2
accelerate: 1.15.0
pandas: 2.3.3
pyarrow: 23.0.1
scikit-learn: 1.6.1
statsmodels: 0.15.0
matplotlib: 3.10.0


### 0.4 Tải tokenizer và model chính
- **Mục đích:** lấy `Qwen/Qwen2.5-1.5B-Instruct` từ Hugging Face Hub. `from_pretrained(tên)` tải file từ `huggingface.co/<tên>` về bộ nhớ đệm, lần sau đọc lại, không tải lại. Model công khai, không cần token.
- **Đầu vào:** `MODEL_INSTRUCT`, `DTYPE`. Cần bật Internet; khoảng 3.1 GB.
- **Đầu ra:** `tok`, `model` (chế độ `eval()`); in `generation_config` để thấy bản Instruct mặc định lấy mẫu ngẫu nhiên.

In [17]:
# 0.4 Tải tokenizer và model chính
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_INSTRUCT)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_INSTRUCT,
    torch_dtype=DTYPE,   # float16
    device_map={"": 0},  # cả model lên GPU 0 (~3,1 GB fp16); "auto" có thể chia sang GPU 1 của T4 x2
)
model.eval()  # chỉ suy luận, không train (tắt dropout)

# Bản Instruct mặc định do_sample=True → mọi lần gọi generate() phải truyền tham số sinh tường minh
print(model.generation_config)
# Mã commit trên Hub để tải đúng bản. transformers 5.x trên Kaggle trả _commit_hash = None → hỏi Hub (cần Internet)
from huggingface_hub import model_info
MODEL_REVISION = getattr(model.config, "_commit_hash", None) or model_info(MODEL_INSTRUCT).sha
assert MODEL_REVISION, "Không lấy được revision của model"
log(f"model chính: {MODEL_INSTRUCT} (revision {MODEL_REVISION}) | device: {model.device}", part="part00_setup", to_run=True)
log(f"generation_config mặc định: do_sample={model.generation_config.do_sample}, "
    f"temperature={model.generation_config.temperature}, top_p={model.generation_config.top_p}, "
    f"top_k={model.generation_config.top_k}, repetition_penalty={model.generation_config.repetition_penalty}",
    part="part00_setup", to_run=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

model chính: Qwen/Qwen2.5-1.5B-Instruct (revision 989aa7980e4cf806f80c7fef2b1adb7bc71aa306) | device: cuda:0
generation_config mặc định: do_sample=True, temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.1


### 0.5 Kiến trúc và số tham số
- **Mục đích:** in thông số kiến trúc từ `model.config` và đếm số tham số bằng code, đối chiếu với model card (`docs/models/qwen2.5-1.5b-instruct_v2.md`: 1,543,714,304).
- **Đầu vào:** `model` (cell 0.4).
- **Đầu ra:** dict `ARCH`; ghi vào log; `assert` số tham số.

In [18]:
# 0.5 Kiến trúc: đọc từ config, không chép tay
cfg = model.config
ARCH = {
    "num_hidden_layers": cfg.num_hidden_layers,        # số decoder block
    "hidden_size": cfg.hidden_size,
    "num_attention_heads": cfg.num_attention_heads,    # số head Q
    "num_key_value_heads": cfg.num_key_value_heads,    # số head K, V (GQA)
    "intermediate_size": cfg.intermediate_size,        # FFN
    "vocab_size": cfg.vocab_size,
    "max_position_embeddings": cfg.max_position_embeddings,
    "tie_word_embeddings": cfg.tie_word_embeddings,
}
# parameters() chỉ trả mỗi tensor một lần, nên embedding dùng chung với LM head được đếm một lần
ARCH["n_params"] = sum(p.numel() for p in model.parameters())

for k, v in ARCH.items():
    log(f"{k}: {v:,}" if isinstance(v, int) and not isinstance(v, bool) else f"{k}: {v}", part="part00_setup", to_run=True)

# Tự kiểm: khớp số đã tính tay từ config.json (model card: 1.54 tỉ)
assert ARCH["n_params"] == 1_543_714_304, ARCH["n_params"]
assert ARCH["num_hidden_layers"] == 28 and ARCH["hidden_size"] == 1536

num_hidden_layers: 28
hidden_size: 1,536
num_attention_heads: 12
num_key_value_heads: 2
intermediate_size: 8,960
vocab_size: 151,936
max_position_embeddings: 32,768
tie_word_embeddings: True
n_params: 1,543,714,304


### 0.6 Tải tokenizer GPT-2
- **Mục đích:** chỉ lấy tokenizer (vài MB, không tải model) để phần 3 so số token với Qwen.
- **Đầu vào:** `TOKENIZER_GPT2`.
- **Đầu ra:** `gpt2_tok`.

In [19]:
# 0.6 Tokenizer GPT-2: byte-level BPE, từ điển 50,257 token
gpt2_tok = AutoTokenizer.from_pretrained(TOKENIZER_GPT2)
log(f"tokenizer phụ: {TOKENIZER_GPT2} | vocab_size: {gpt2_tok.vocab_size:,}", part="part00_setup", to_run=True)
assert gpt2_tok.vocab_size == 50_257

tokenizer phụ: openai-community/gpt2 | vocab_size: 50,257


### 0.7 Tải dữ liệu UIT-VSFC
- **Mục đích:** đọc 3 split (train, validation, test) dạng Parquet bằng `pandas`; bỏ nhãn neutral để còn bài toán 2 nhãn cho phần 7, 10.
- **Đầu vào:** `UIT_VSFC_BASE` (Hugging Face). Cột: `sentence`, `sentiment` (0 negative, 1 neutral, 2 positive), `topic`.
- **Đầu ra:** `vsfc` (dict 3 DataFrame, đã bỏ neutral), `LABEL_NAMES`. Dữ liệu không lưu vào repo.

In [20]:
# 0.7 UIT-VSFC: đọc Parquet thẳng từ Hugging Face
import pandas as pd

EXPECTED_ROWS = {"train": 11_426, "validation": 1_583, "test": 3_166}  # theo trang dataset
LABEL_NAMES = {0: "negative", 2: "positive"}  # bỏ 1 = neutral

vsfc: dict[str, pd.DataFrame] = {}
for split, n_expected in EXPECTED_ROWS.items():
    df = pd.read_parquet(f"{UIT_VSFC_BASE}/{split}/0000.parquet")
    assert len(df) == n_expected, (split, len(df))           # tải đủ, đúng bản
    assert {"sentence", "sentiment"} <= set(df.columns)
    df = df[df["sentiment"].isin(LABEL_NAMES)].reset_index(drop=True)  # bỏ neutral
    vsfc[split] = df
    counts = df["sentiment"].map(LABEL_NAMES).value_counts().to_dict()
    log(f"UIT-VSFC {split}: {n_expected:,} dòng gốc → {len(df):,} dòng sau khi bỏ neutral {counts}",
        part="part00_setup", to_run=True)

UIT-VSFC train: 11,426 dòng gốc → 10,968 dòng sau khi bỏ neutral {'positive': 5643, 'negative': 5325}
UIT-VSFC validation: 1,583 dòng gốc → 1,510 dòng sau khi bỏ neutral {'positive': 805, 'negative': 705}
UIT-VSFC test: 3,166 dòng gốc → 2,999 dòng sau khi bỏ neutral {'positive': 1590, 'negative': 1409}


### 0.8 Lưu `env.json`, `config.json` và kết thúc phần 0
- **Mục đích:** lưu môi trường và tham số để chép lên slide 2 và để người khác chạy lại đúng cấu hình.
- **Đầu vào:** `ENV`, `ARCH` và các hằng số ở cell 0.1.
- **Đầu ra:** `RESULTS_DIR/env.json`; `LOG_DIR/env.json`, `LOG_DIR/config.json`; dòng tổng kết trong `run.log` và `part00_setup.log`.

In [21]:
# 0.8 Ghi file JSON và tổng kết phần 0
import json

CONFIG = {
    "run_id": RUN_ID, "smoke": SMOKE, "seed": SEED, "dtype": ENV["dtype"],
    "model_instruct": MODEL_INSTRUCT, "model_instruct_revision": MODEL_REVISION,
    "model_base": MODEL_BASE, "tokenizer_gpt2": TOKENIZER_GPT2,
    "n_test": N_TEST, "uit_vsfc_base": UIT_VSFC_BASE,
    "architecture": ARCH,
}


def save_json(obj: dict, path: Path) -> None:
    """Ghi dict ra file JSON (UTF-8, giữ chữ tiếng Việt)."""
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


save_json(ENV, RESULTS_DIR / "env.json")
save_json(ENV, LOG_DIR / "env.json")
save_json(CONFIG, LOG_DIR / "config.json")

# Tự kiểm: đọc lại được
assert json.loads((RESULTS_DIR / "env.json").read_text(encoding="utf-8"))["run_id"] == RUN_ID
assert json.loads((LOG_DIR / "config.json").read_text(encoding="utf-8"))["seed"] == SEED

log(f"=== PART 00 setup END {now_iso()} ===", part="part00_setup")
log(f"[part00_setup] OK {now_iso()} → {(RESULTS_DIR / 'env.json').relative_to(ROOT)}, "
    f"{(LOG_DIR / 'config.json').relative_to(ROOT)}")

=== PART 00 setup END 2026-11-16T03:04:10+07:00 ===
[part00_setup] OK 2026-11-16T03:04:10+07:00 → outputs/_smoke/results/env.json, outputs/_smoke/logs/20261116_030403_smoke/config.json


**Phân tích (smoke run 20261116_025328_smoke và 20261116_030403_smoke, Kaggle):**
- Môi trường thật: Python 3.13.16, torch 2.11.0+cu130 (CUDA 13.0), transformers 5.20.2, accelerate 1.15.0; GPU Tesla T4, VRAM 14.6 GB, dtype float16 (từ `env.json`).
- Số tham số đếm bằng code 1,543,714,304 so với 1,543,714,304 của model card: khớp. Kiến trúc từ `model.config`: 28 block, hidden 1536, 12 head Q / 2 head KV (GQA), FFN 8960, vocab 151,936, context 32,768, embedding dùng chung với LM head.
- UIT-VSFC: số dòng gốc khớp (11,426 / 1,583 / 3,166); bỏ neutral còn 10,968 / 1,510 / 2,999. Test: 1,590 positive / 1,409 negative (53% / 47%), gần cân bằng.
- `generation_config` mặc định của bản Instruct: `do_sample=True`, temperature 0.7, top_p 0.8, top_k 20, repetition_penalty 1.1 → nếu không truyền tham số, mỗi lần chạy cho kết quả khác nhau; các phần sau luôn truyền tham số sinh tường minh (đánh giá dùng `do_sample=False`).
- Model đã tải: revision `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`, nằm trên `cuda:0`; Kaggle cấp 2 GPU (`gpu_count` = 2), model chỉ dùng GPU 0.
- Ghi chú: lần chạy 025328 chưa có `gpu_count` và log `revision None`; đã sửa cell 0.3, 0.4, lần chạy 030403 có đủ.


## Phần 1 — Tác vụ mẫu (mục 1, lý thuyết: Muội; demo + code: Toàn)

- Desc: `docs/parts/muc01_gioi_thieu_v2.md`. Một model (Qwen2.5-1.5B-Instruct), cách sinh greedy, chỉ đổi prompt: hỏi đáp, tóm tắt, dịch, sinh mã.
- Đầu vào: `data/tasks_prompts.json`. Đầu ra: `results/tasks_outputs.csv`; log `part01_tasks.log`, `part01_tasks_records.csv`.
- Gồm 5 cell code: 1.1 hàm sinh, 1.2 chạy 4 tác vụ, 1.3 chạy code do model sinh, 1.4 chấm và ghi kết quả, 1.5 thử prompt khác (demo trực tiếp, không chấm).

### 1.1 Hàm sinh cho bản Instruct
- **Mục đích:** một hàm dùng chung cho mọi lần sinh ở phần 1: định dạng prompt bằng chat template, sinh greedy, cắt phần prompt khỏi output.
- **Đầu vào:** `model`, `tok` (cell 0.4); chuỗi prompt; `max_new_tokens`.
- **Đầu ra:** hàm `show_io()` (in INPUT và OUTPUT); hàm `chat_generate()` trả (output, số token vào, số token ra, thời gian giây); dòng START và tham số trong `part01_tasks.log`.

In [ ]:
# 1.1 Hàm sinh: chat template + greedy
import time

PART01 = "part01_tasks"
log(f"=== PART 01 tasks START {now_iso()} ===", part=PART01)


def chat_generate(model, tok, prompt: str, max_new_tokens: int) -> tuple[str, int, int, float]:
    """Sinh greedy cho một tin nhắn người dùng; trả (output, số token vào, số token ra, giây)."""
    messages = [{"role": "user", "content": prompt}]
    # add_generation_prompt=True: thêm phần mở đầu lượt "assistant" để model trả lời
    enc = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = enc["input_ids"].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,                       # greedy: mỗi bước lấy token xác suất cao nhất
            temperature=None, top_p=None, top_k=None,  # tắt tham số lấy mẫu mặc định của bản Instruct
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    new_ids = out[0, n_in:]  # chỉ giữ phần model sinh thêm (tính cả token kết thúc nếu có)
    text = tok.decode(new_ids, skip_special_tokens=True).strip()
    return text, n_in, len(new_ids), elapsed


def show_io(title: str, prompt: str, output: str, n_in: int, n_out: int, max_new_tokens: int, sec: float) -> None:
    """In rõ INPUT (prompt người dùng) và OUTPUT (câu trả lời của model)."""
    cut = "  ← BỊ CẮT ở max_new_tokens" if n_out >= max_new_tokens else ""
    print(f"==================== {title} ====================")
    print(f"INPUT  (prompt; {n_in} token sau chat template, gồm cả system prompt mặc định):")
    print(prompt)
    print(f"\nOUTPUT ({n_out}/{max_new_tokens} token, {sec:.2f} s){cut}:")
    print(output)
    print()


# Ghi một ví dụ prompt sau khi định dạng, để thấy chat template thêm gì (system prompt mặc định của Qwen)
demo_messages = [{"role": "user", "content": "Xin chào"}]
log("prompt sau chat template (ví dụ 'Xin chào'):\n"
    + tok.apply_chat_template(demo_messages, add_generation_prompt=True, tokenize=False), part=PART01)

# Tự kiểm: sinh được chuỗi không rỗng
test_out, test_in, test_n, test_s = chat_generate(model, tok, "Xin chào", max_new_tokens=16)
show_io("Thử hàm", "Xin chào", test_out, test_in, test_n, 16, test_s)
assert isinstance(test_out, str) and test_out

### 1.2 Chạy 4 tác vụ
- **Mục đích:** cùng model, cùng cách sinh, chỉ đổi prompt → 4 tác vụ khác nhau.
- **Đầu vào:** `DATA_DIR / "tasks_prompts.json"` (`id, task, prompt, max_new_tokens, check_method, expected`).
- **Đầu ra:** `records_df` (4 dòng); `LOG_DIR/part01_tasks_records.csv`; mỗi tác vụ một dòng trong `part01_tasks.log`.

In [ ]:
# 1.2 Chạy 4 tác vụ, mỗi tác vụ ghi một dòng log
import csv
import io
import json

tasks_file = DATA_DIR / "tasks_prompts.json"
TASKS_EXPECTED_VERSION = "v2"  # đổi khi sửa data/tasks_prompts.json
tasks_json = json.loads(tasks_file.read_text(encoding="utf-8"))
assert tasks_json["_meta"]["version"] == TASKS_EXPECTED_VERSION, (
    f"tasks_prompts.json đang là {tasks_json['_meta']['version']}, cần {TASKS_EXPECTED_VERSION}: "
    "cập nhật Kaggle Dataset (New Version) và gắn lại vào notebook")
TASKS: list[dict] = tasks_json["tasks"]
assert [t["id"] for t in TASKS] == ["qa", "summary", "translate", "code"], [t["id"] for t in TASKS]

REC_COLS = ["idx", "id", "task", "prompt", "n_input_tokens", "n_output_tokens", "truncated", "output", "time_s"]


def csv_line(values: list) -> str:
    """Một dòng CSV (tự thêm dấu nháy khi chuỗi có dấu phẩy, xuống dòng)."""
    buf = io.StringIO()
    csv.writer(buf, lineterminator="").writerow(values)
    return buf.getvalue()


log(f"tham số: model={MODEL_INSTRUCT} | do_sample=False | "
    f"max_new_tokens={ {t['id']: t['max_new_tokens'] for t in TASKS} } | "
    f"repetition_penalty={model.generation_config.repetition_penalty} (mặc định của model)", part=PART01)
log(f"dữ liệu vào: data/tasks_prompts.json {TASKS_EXPECTED_VERSION} ({len(TASKS)} tác vụ)", part=PART01)
log("--- mỗi mẫu ---", part=PART01, echo=False)
log(",".join(REC_COLS), part=PART01, echo=False)

records = []
for idx, t in enumerate(TASKS):
    output, n_in, n_out, sec = chat_generate(model, tok, t["prompt"], t["max_new_tokens"])
    rec = {"idx": idx, "id": t["id"], "task": t["task"], "prompt": t["prompt"],
           "n_input_tokens": n_in, "n_output_tokens": n_out,
           "truncated": n_out >= t["max_new_tokens"],  # chạm trần → output bị cắt
           "output": output, "time_s": round(sec, 2)}
    records.append(rec)
    log(csv_line([rec[c] for c in REC_COLS]), part=PART01, echo=False)
    show_io(t["task"], t["prompt"], output, n_in, n_out, t["max_new_tokens"], sec)

records_df = pd.DataFrame(records, columns=REC_COLS)
records_df.to_csv(LOG_DIR / f"{PART01}_records.csv", index=False, encoding="utf-8")

# Tự kiểm
assert len(records_df) == 4
assert records_df["output"].str.len().gt(0).all()
for t, n in zip(TASKS, records_df["n_output_tokens"]):
    assert n <= t["max_new_tokens"], (t["id"], n)

### 1.3 Chạy hàm `is_even` do model sinh ra
- **Mục đích:** kiểm tác vụ sinh mã bằng cách chạy thật. **Code dưới đây lấy từ output của model, không phải code của nhóm.**
- **Đầu vào:** output tác vụ `code` (cell 1.2).
- **Đầu ra:** `IS_EVEN_RESULTS` (kết quả với n = 0, 1, 2, −3) hoặc `IS_EVEN_ERROR`.
- **Lưu ý an toàn:** `exec` trong namespace riêng chỉ tránh đụng biến của notebook, **không phải sandbox**. Đọc code model in ra trước khi chạy normal run.

In [ ]:
# 1.3 Tách code từ output của model rồi chạy thử
import re


def extract_code(text: str) -> str:
    """Lấy khối ```python ...``` đầu tiên; không có khối thì trả cả chuỗi."""
    m = re.search(r"```(?:python|py)?[ \t]*\n(.*?)```", text, flags=re.DOTALL)
    return m.group(1) if m else text


code_task = next(t for t in TASKS if t["id"] == "code")
model_code = extract_code(records_df.loc[records_df["id"] == "code", "output"].iloc[0])
print("=== Code do MODEL sinh ===")
print(model_code)

IS_EVEN_INPUTS = [0, 1, 2, -3]
IS_EVEN_RESULTS: list | None = None
IS_EVEN_ERROR: str | None = None
ns: dict = {}  # namespace riêng cho code của model
try:
    exec(model_code, ns)
    IS_EVEN_RESULTS = [ns["is_even"](n) for n in IS_EVEN_INPUTS]
except Exception as e:  # code của model có thể sai cú pháp / thiếu hàm: ghi lỗi, không dừng notebook
    IS_EVEN_ERROR = f"{type(e).__name__}: {e}"

log(f"is_even{IS_EVEN_INPUTS} → {IS_EVEN_RESULTS} | kỳ vọng {code_task['expected']} | lỗi: {IS_EVEN_ERROR}",
    part=PART01)

### 1.4 Chấm và ghi `tasks_outputs.csv`
- **Mục đích:** chấm 4 tác vụ theo thang ghi sẵn trong desc (mục Đánh giá) và `check_method` của từng tác vụ.
- **Đầu vào:** `records_df` (1.2), `IS_EVEN_RESULTS` / `IS_EVEN_ERROR` (1.3), `MANUAL_JUDGE` (Toàn điền sau khi đọc output tóm tắt).
- **Đầu ra:** `RESULTS_DIR/tasks_outputs.csv` (cột `id, task, prompt, output, n_input_tokens, n_output_tokens, truncated, time_s, check_method, judged, judge_note`); dòng END + tổng kết trong log.
- Tóm tắt chấm tay: đọc output ở 1.2, điền `MANUAL_JUDGE["summary"]`. Smoke run cho phép để `None` (ghi "chưa chấm"); normal run bắt buộc đã chấm.

In [ ]:
# 1.4 Chấm 4 tác vụ
# Chấm tay (Toàn): ("đạt" | "không đạt", lý do theo 3 điều kiện: 1 câu, giữ ý chính, không thêm ý)
MANUAL_JUDGE: dict[str, tuple[str, str] | None] = {
    "summary": ("đạt", "đúng 1 câu, giữ đủ 5 ý, không thêm ý; nhưng gần như ghép lại các câu gốc, rút gọn ít"),
}


def judge(task_id: str, output: str) -> tuple[str, str]:
    """Chấm một tác vụ; trả (judged, judge_note)."""
    low = output.lower()
    if task_id == "qa":
        ok = "tokyo" in low
        return ("đạt" if ok else "không đạt", "có 'Tokyo'" if ok else "không có 'Tokyo'")
    if task_id == "translate":
        has_today = "today" in low
        has_word = any(w in low for w in ("weather", "nice", "beautiful", "lovely"))
        ok = has_today and has_word
        return ("đạt" if ok else "không đạt", f"today={has_today}, weather/nice/beautiful/lovely={has_word}")
    if task_id == "code":
        if IS_EVEN_ERROR:
            return "không đạt", f"lỗi khi chạy code của model: {IS_EVEN_ERROR}"
        ok = IS_EVEN_RESULTS == code_task["expected"]
        note = f"is_even{IS_EVEN_INPUTS} = {IS_EVEN_RESULTS}"
        # Ghi nhận (không trừ điểm, prompt không cấm): model viết thêm chữ ngoài khối code
        if re.sub(r"```.*?```", "", output, flags=re.DOTALL).strip():
            note += " | tự thêm lời dẫn / giải thích ngoài khối code (prompt không cấm, không trừ điểm)"
        return ("đạt" if ok else "không đạt", note)
    if task_id == "summary":
        manual = MANUAL_JUDGE["summary"]
        return manual if manual else ("chưa chấm", "chấm tay: chưa điền MANUAL_JUDGE")
    raise ValueError(task_id)


check_map = {t["id"]: t["check_method"] for t in TASKS}
results_df = records_df.drop(columns=["idx"]).copy()
results_df["check_method"] = results_df["id"].map(check_map)
results_df[["judged", "judge_note"]] = [judge(i, o) for i, o in zip(results_df["id"], results_df["output"])]
# Output bị cắt: ghi rõ trong lý do (chấm trên phần đã sinh, không đoán phần bị cắt)
results_df.loc[results_df["truncated"], "judge_note"] += " | bị cắt ở max_new_tokens"
results_df.to_csv(RESULTS_DIR / "tasks_outputs.csv", index=False, encoding="utf-8")
print(results_df[["task", "judged", "judge_note"]].to_string(index=False))

# Tự kiểm: đủ 4 dòng, nhãn chấm hợp lệ; normal run phải chấm xong cả tóm tắt
allowed = {"đạt", "không đạt"} | ({"chưa chấm"} if SMOKE else set())
assert len(results_df) == 4 and set(results_df["judged"]) <= allowed, results_df["judged"].tolist()

n_pass = int((results_df["judged"] == "đạt").sum())
log(f"=== PART 01 tasks END {now_iso()} ===", part=PART01)
log(f"tổng kết: đạt {n_pass}/4 | {dict(zip(results_df['id'], results_df['judged']))} | "
    f"bị cắt: {int(results_df['truncated'].sum())}/4 | thời gian sinh {records_df['time_s'].sum():.1f} s", part=PART01)
log(f"[{PART01}] OK {now_iso()} → {(RESULTS_DIR / 'tasks_outputs.csv').relative_to(ROOT)} (4 dòng)")

**Phân tích (smoke run 16/11, Kaggle T4; slide lấy số từ normal run):**
- Cùng một model, cùng cách sinh greedy, chỉ đổi prompt là model chuyển từ hỏi đáp sang tóm tắt, dịch, viết code: đạt 4/4 theo thang chấm ghi trước khi chạy, không output nào bị cắt.
- Chưa hoàn hảo: tóm tắt gần như ghép lại 5 câu gốc thành 1 câu dài (104 token), rút gọn ít; sinh mã đúng nhưng tự thêm ví dụ `print` và giải thích dù không được yêu cầu (216 token); prompt thử "Xin chào" ở 1.1 được trả lời bằng tiếng Trung.
- Kết luận: LLM là model đa tác vụ nhưng vẫn phải kiểm output (liên hệ mục 5 về hành vi bản Instruct, mục 9 về hạn chế).


### 1.5 Thử prompt khác (demo trực tiếp)
- **Mục đích:** khi thuyết trình, gõ câu khác ngay trong cell để cô xem model trả lời; dùng cùng hàm, cùng cách sinh greedy với 1.2.
- **Cách dùng:** sửa `MY_PROMPT` (và `MY_MAX_NEW_TOKENS` nếu cần) rồi chạy lại **riêng cell này** (Shift + Enter). Chạy bao nhiêu lần cũng được.
- **Đầu vào:** `MY_PROMPT`, `MY_MAX_NEW_TOKENS`; `model`, `tok`, `chat_generate()`, `show_io()` (cell 1.1).
- **Đầu ra:** INPUT / OUTPUT in ngay dưới cell; mỗi lần chạy thêm 1 dòng vào `LOG_DIR/part01_custom.log`.
- Cell này **không chấm** và không ghi vào `tasks_outputs.csv`: kết quả chấm ở 1.4 chỉ dựa trên 4 prompt cố định trong `data/tasks_prompts.json`.

In [ ]:
# 1.5 Thử prompt khác: sửa 2 dòng dưới rồi chạy lại riêng cell này
MY_PROMPT = "Giải thích ngắn gọn mạng nơ-ron nhân tạo là gì, trong 2 câu."
MY_MAX_NEW_TOKENS = 256

my_out, my_in, my_n, my_s = chat_generate(model, tok, MY_PROMPT, MY_MAX_NEW_TOKENS)
show_io("Thử prompt khác", MY_PROMPT, my_out, my_in, my_n, MY_MAX_NEW_TOKENS, my_s)
log(csv_line([now_iso(), MY_PROMPT, my_in, my_n, my_n >= MY_MAX_NEW_TOKENS, my_out, round(my_s, 2)]),
    part="part01_custom", echo=False)

## Phần 2 — Kiến trúc (mục 2, lý thuyết: Muội; code: Toàn)

- Nội dung: Thông số model, causal mask, trọng số attention thật (eager), top-5 token tiếp theo
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 3 — Tokenization (mục 3, lý thuyết: Muội; code: Toàn)

- Nội dung: Tokenize câu Việt và Anh; so số token GPT-2 và Qwen; embedding
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 4 — Nguyên lý học (mục 4, lý thuyết: Phúc; code: Toàn)

- Nội dung: Cặp ngữ cảnh → token mục tiêu; loss từng token; so với outputs.loss
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 5 — Base vs Instruct (mục 5, lý thuyết: Phúc; code: Toàn)

- Nội dung: Cùng prompt trên 2 bản, cùng do_sample=False (chờ cô xác nhận)
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 6 — Sinh văn bản (mục 6, lý thuyết: Phúc; code: Toàn)

- Nội dung: Sinh từng bước; 5 cấu hình decoding, đặt đủ tham số
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 7 — Zero-shot và few-shot (mục 7, lý thuyết: Toàn; code: Toàn)

- Nội dung: UIT-VSFC 500 câu (250/250); zero-shot vs 3-shot × 3 bộ; greedy (chờ cô xác nhận)
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 8 — RAG (mục 8, lý thuyết: Muội; code: Toàn)

- Nội dung: Chia đoạn slide môn học, TF-IDF, không RAG vs có RAG (chờ cô xác nhận)
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 9 — Hạn chế (mục 9, lý thuyết: Phúc; code: Toàn)

- Nội dung: Hallucination (~20 câu có đáp án), thiên lệch nghề nghiệp, ngữ cảnh dài
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._

## Phần 10 — Tổng hợp (mục 10, lý thuyết: Toàn; code: Toàn)

- Nội dung: Accuracy, confusion matrix, McNemar; phân tích lỗi; bảng tổng hợp mọi phần
- Kết quả ghi vào: xem `project_description.md` §2.3

_Chưa có code._